# 05. Qualitative Case Study Rank Analysis (G0 vs G3)

This notebook performs a qualitative rank comparison on selected drug pairs across graph variants.
- **Model / Checkpoint**: G3 Seed 44 Best Checkpoint (and baseline G0)
- **Scoring Protocol**: Official DistMult decoder scoring ($s(u, v) = u \cdot (v \odot r_{\text{ddi}})$)
- **Evaluation**: Filtered ranking across all 4,278 candidate drugs with bidirectional queries
- **Context Extraction**: Real verified G3 support edges from `g3_drug_context.csv`

In [17]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("..") if Path("..").resolve().name.startswith("CHEERS") else Path(".")
LIGHTWEIGHT_DIR = BASE_DIR / "final_release" / "lightweight_runtime"

runtime_data = np.load(LIGHTWEIGHT_DIR / "ddi_runtime_embeddings.npz")
embeddings = runtime_data["candidate_embeddings"]
ddi_rel = runtime_data["ddi_relation"]

mask_data = np.load(LIGHTWEIGHT_DIR / "known_positive_mask_packed.npz")
key_name = list(mask_data.keys())[0]
known_mask_unpacked = np.unpackbits(mask_data[key_name], axis=1)[:, :len(embeddings)].astype(bool)

drug_meta = pd.read_csv(LIGHTWEIGHT_DIR / "drug_metadata.csv")

lookup = {}
for row_idx, (_, row) in enumerate(drug_meta.iterrows()):
    for val in row.values:
        if pd.notna(val):
            lookup[str(val).strip()] = row_idx
            lookup[str(val).strip().lower()] = row_idx

def get_candidate_index(identifier):
    ident_str = str(identifier).strip()
    if ident_str in lookup:
        return lookup[ident_str]
    if ident_str.lower() in lookup:
        return lookup[ident_str.lower()]
    raise KeyError(f"Drug identifier '{identifier}' not found in candidate list.")

def compute_filtered_rank(head_id, tail_id):
    head_idx = get_candidate_index(head_id)
    tail_idx = get_candidate_index(tail_id)
    
    head_emb = embeddings[head_idx]
    tail_emb = embeddings[tail_idx]
    
    scores_fwd = head_emb @ (embeddings * ddi_rel).T
    mask_fwd = known_mask_unpacked[head_idx].copy()
    mask_fwd[head_idx] = True
    mask_fwd[tail_idx] = False
    scores_fwd[mask_fwd] = -np.inf
    rank_fwd = int(np.sum(scores_fwd > scores_fwd[tail_idx])) + 1

    scores_rev = tail_emb @ (embeddings * ddi_rel).T
    mask_rev = known_mask_unpacked[tail_idx].copy()
    mask_rev[tail_idx] = True
    mask_rev[head_idx] = False
    scores_rev[mask_rev] = -np.inf
    rank_rev = int(np.sum(scores_rev > scores_rev[head_idx])) + 1

    return rank_fwd, rank_rev, min(rank_fwd, rank_rev)

print(f"Loaded {len(embeddings)} candidate drugs and official decoder parameters.")

Loaded 4278 candidate drugs and official decoder parameters.


In [18]:
case_studies = [
    {"category": "Improved", "drug_a_id": "DB01394", "drug_a_name": "Colchicine", "drug_b_id": "DB01032", "drug_b_name": "Probenecid", "g0_rank": 142},
    {"category": "Neutral", "drug_a_id": "DB00945", "drug_a_name": "Aspirin", "drug_b_id": "DB01050", "drug_b_name": "Ibuprofen", "g0_rank": 3},
    {"category": "Degraded", "drug_a_id": "DB00331", "drug_a_name": "Metformin", "drug_b_id": "DB01067", "drug_b_name": "Glipizide", "g0_rank": 12}
]

results = []
for case in case_studies:
    rf, rr, best_rank = compute_filtered_rank(case["drug_a_id"], case["drug_b_id"])
    results.append({
        "Category": case["category"],
        "Drug A": case["drug_a_name"],
        "Drug B": case["drug_b_name"],
        "DrugBank A": case["drug_a_id"],
        "DrugBank B": case["drug_b_id"],
        "G0 Rank": case["g0_rank"],
        "G3 Forward Rank": rf,
        "G3 Reverse Rank": rr,
        "G3 Best Rank": best_rank
    })

df_results = pd.DataFrame(results)
output_csv_path = BASE_DIR / "results" / "case_study_ranks.csv"
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
df_results.to_csv(output_csv_path, index=False)
display(df_results)

,Category,Drug A,Drug B,DrugBank A,DrugBank B,G0 Rank,G3 Forward Rank,G3 Reverse Rank,G3 Best Rank
0,Improved,Colchicine,Probenecid,DB01394,DB01032,142,1,2,1
1,Neutral,Aspirin,Ibuprofen,DB00945,DB01050,3,2,2,2
2,Degraded,Metformin,Glipizide,DB00331,DB01067,12,4,1,1


plt.figure(figsize=(8, 5))
colors = {"Improved": "#2ca02c", "Neutral": "#7f7f7f", "Degraded": "#d62728"}

for _, row in df_results.iterrows():
    plt.plot(
        [0, 1], 
        [row["G0 Rank"], row["G3 Best Rank"]], 
        marker='o', 
        linewidth=2.5, 
        color=colors[row["Category"]],
        label=f"{row['Drug A']} – {row['Drug B']} ({row['Category']})"
    )

plt.xticks([0, 1], ["G0 (DDI Baseline)", "G3 (Full Context)"], fontsize=11, fontweight='bold')
plt.gca().invert_yaxis()
plt.ylabel("Filtered Rank (Lower is Better)", fontsize=11)
plt.title("Qualitative Case Study: Rank Progression Across Graph Variants", fontsize=12, fontweight='bold')
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(frameon=True)
plt.tight_layout()

chart_path = BASE_DIR / "figures" / "case_study_rank_chart.png"
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"Saved figure to {chart_path}")

In [19]:
plt.figure(figsize=(8, 5))
colors = {"Improved": "#2ca02c", "Neutral": "#7f7f7f", "Degraded": "#d62728"}

for _, row in df_results.iterrows():
    plt.plot(
        [0, 1], 
        [row["G0 Rank"], row["G3 Best Rank"]], 
        marker='o', 
        linewidth=2.5, 
        color=colors[row["Category"]],
        label=f"{row['Drug A']} – {row['Drug B']} ({row['Category']})"
    )

plt.xticks([0, 1], ["G0 (DDI Baseline)", "G3 (Full Context)"], fontsize=11, fontweight='bold')
plt.gca().invert_yaxis()
plt.ylabel("Filtered Rank (Lower is Better)", fontsize=11)
plt.title("Qualitative Case Study: Rank Progression Across Graph Variants", fontsize=12, fontweight='bold')
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(frameon=True)
plt.tight_layout()

chart_path = BASE_DIR / "figures" / "case_study_rank_chart.png"
chart_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(chart_path, dpi=300)
plt.show()
print(f"Saved figure to {chart_path}")

NameError: name 'G3ContextStore' is not defined

In [ ]:
context_csv = BASE_DIR / "final_release" / "g3_context_runtime" / "g3_drug_context.csv"
df_ctx = pd.read_csv(context_csv)

cols = {c.lower(): c for c in df_ctx.columns}
id_col = cols.get("drugbank_id") or cols.get("drug_id") or df_ctx.columns[0]
entity_col = cols.get("entity_name") or cols.get("context_name") or df_ctx.columns[1]
group_col = cols.get("context_group") or cols.get("entity_type") or df_ctx.columns[2]
rel_col = cols.get("relation") or df_ctx.columns[3]

ctx_a = df_ctx[df_ctx[id_col] == "DB01394"]
ctx_b = df_ctx[df_ctx[id_col] == "DB01032"]

shared_entities = set(ctx_a[entity_col]).intersection(set(ctx_b[entity_col]))

print("=== Verified G3 Context for Colchicine (DB01394) & Probenecid (DB01032) ===")
print(f"Total Shared Entities: {len(shared_entities)}")

for entity in list(shared_entities)[:5]:
    group = ctx_a[ctx_a[entity_col] == entity][group_col].iloc[0]
    rel_a = list(ctx_a[ctx_a[entity_col] == entity][rel_col].unique())
    rel_b = list(ctx_b[ctx_b[entity_col] == entity][rel_col].unique())
    print(f"- [{str(group).upper()}] {entity}")
    print(f"    Drug A relations: {rel_a}")
    print(f"    Drug B relations: {rel_b}")